---
# MATERIAL COMPOUND CLASSIFIER
---
#### Description: 
A helper for PCA analysis to classify the complete data into chemistry-based sheets.

Classifies materials into compound categories based on elemental
composition. 
Accepts either a 'species' column (list of symbols)
or a 'formula'/'FORMULA' column (chemical formula string like
Fe2O3, MoSi2, NaCl).

#### Categories (priority order):
  1. Element        — single element only
  2. Carbide        — metal/metalloid + C
  3. Boride         — metal/metalloid + B
  4. Nitride        — metal/metalloid + N
  5. Oxide          — contains O
  6. Halide         — contains any halogen
  7. Metal-Metalloid — metals only OR metals + metalloids, no true non-metals
  8. Other          — everything else

#### Note: 

Metals and metalloids are grouped together for Carbide/Boride/Nitride because both form similar solid-solution and interstitial compounds.


In [7]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import sys
import pandas as pd
from pathlib import Path
from typing import Set, Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# TERMINAL COLOR CODES
# ============================================================================
class Colors:
    BOLD    = '\033[1m'
    ENDC    = '\033[0m'

    RED          = '\033[91m'
    ORANGE       = '\033[38;5;214m'
    YELLOW       = '\033[93m'
    PINK         = '\033[38;5;213m'
    LIME         = '\033[38;5;154m'
    CYAN         = '\033[96m'
    DARK_BLUE    = '\033[34m'
    WHITE        = '\033[97m'
    GREY         = '\033[90m'

    HEADER  = '\033[95m'
    BLUE    = '\033[94m'
    GREEN   = '\033[92m'

    try:
        '█'.encode(sys.stdout.encoding or 'utf-8')
        BAR_CHAR = '█'
    except (UnicodeEncodeError, LookupError):
        BAR_CHAR = '#'

    @classmethod
    def header(cls, text):
        print(f"\n{cls.HEADER}{cls.BOLD}{'='*80}\n{text.center(80)}\n{'='*80}{cls.ENDC}\n")

    @classmethod
    def step(cls, num, text):
        print(f"\n{cls.BLUE}{cls.BOLD}[STEP {num}] {text}\n{'-'*80}{cls.ENDC}")

    @classmethod
    def success(cls, text): print(f"{cls.GREEN}  ✓  {text}{cls.ENDC}")
    @classmethod
    def warning(cls, text): print(f"{cls.YELLOW}  ⚠  {text}{cls.ENDC}")
    @classmethod
    def error(cls, text):   print(f"{cls.RED}  ✗  {text}{cls.ENDC}")
    @classmethod
    def info(cls, text):    print(f"{cls.CYAN}  ℹ  {text}{cls.ENDC}")
    @classmethod
    def bold(cls, text):    print(f"{cls.BOLD}{text}{cls.ENDC}")


# ============================================================================
# ELEMENT DATABASE
# ============================================================================
class ElementDatabase:
    ALL_METALS: Set[str] = {
        'Li','Na','K','Rb','Cs','Fr',
        'Be','Mg','Ca','Sr','Ba','Ra',
        'Sc','Ti','V','Cr','Mn','Fe','Co','Ni','Cu','Zn',
        'Y','Zr','Nb','Mo','Tc','Ru','Rh','Pd','Ag','Cd',
        'Hf','Ta','W','Re','Os','Ir','Pt','Au','Hg',
        'Al','Ga','In','Sn','Tl','Pb','Bi',
        'Rf','Db','Sg','Bh','Hs','Mt','Ds','Rg','Cn','Nh','Fl','Mc','Lv',
        'La','Ce','Pr','Nd','Pm','Sm','Eu','Gd',
        'Tb','Dy','Ho','Er','Tm','Yb','Lu',
        'Ac','Th','Pa','U','Np','Pu','Am','Cm',
        'Bk','Cf','Es','Fm','Md','No','Lr',
    }

    METALLOIDS: Set[str] = {'Si', 'Ge', 'As', 'Sb', 'Te', 'Po'}
    METALS_AND_METALLOIDS: Set[str] = ALL_METALS | METALLOIDS
    HALOGENS: Set[str] = {'F', 'Cl', 'Br', 'I', 'At', 'Ts'}

    # All known element symbols — used to validate tokens from formula parsing
    ALL_ELEMENTS: Set[str] = METALS_AND_METALLOIDS | HALOGENS | {
        'H', 'He', 'B', 'C', 'N', 'O', 'P', 'S', 'Se',
        'Ne', 'Ar', 'Kr', 'Xe', 'Rn', 'Og',
    }


# ============================================================================
# FORMULA PARSER
# ============================================================================
class FormulaParser:
    """
    Extracts the set of element symbols from a chemical formula string.

    Handles:
      - Standard formulas:  Fe2O3, NaCl, MoSi2, Al2O3
      - Hill-order:         C2H6O
      - Parentheses:        Ca3(PO4)2
      - Whitespace/spaces:  Fe 2 O 3  (Materials Project style)
      - Reduced formulas:   same as above

    Returns an empty set on parse failure.
    """

    # Regex: one uppercase letter optionally followed by one lowercase letter,
    # then optional digits (including decimals for fractional occupancy).
    _TOKEN_RE = re.compile(r'([A-Z][a-z]?)(\d*\.?\d*)')

    def __init__(self):
        self._db = ElementDatabase()

    def parse(self, formula) -> Set[str]:
        if pd.isna(formula):
            return set()

        raw = str(formula).strip()

        # ── Path 1: species-list format  ['Fe', 'C']  or  Fe, C ──────
        if ',' in raw or raw.startswith('['):
            cleaned = raw.strip("[]")
            tokens = {x.strip().strip("'\"") for x in cleaned.split(",") if x.strip()}
            # Validate all tokens look like element symbols
            if all(re.fullmatch(r'[A-Z][a-z]?', t) for t in tokens):
                return tokens

        # ── Path 2: chemical formula string ───────────────────────────
        # Strip parentheses multipliers by removing digits after ')'
        # then extract all [A-Z][a-z]? tokens.
        elements: Set[str] = set()
        for match in self._TOKEN_RE.finditer(raw):
            sym = match.group(1)
            # Two-letter symbol first; if not recognised, try single letter
            if sym in self._db.ALL_ELEMENTS:
                elements.add(sym)
            elif sym[0] in self._db.ALL_ELEMENTS:
                elements.add(sym[0])
            else:
                Colors.warning(f"Unrecognised token '{sym}' in formula: {raw!r}")

        return elements


# ============================================================================
# FILE MANAGER
# ============================================================================
class FileManager:
    SUPPORTED_EXT = ['.xlsx', '.xls', '.csv']

    @staticmethod
    def list_files(folder: str) -> List[Tuple[int, str, str]]:
        files = sorted(
            [f for f in Path(folder).iterdir()
             if f.suffix.lower() in FileManager.SUPPORTED_EXT
             and not f.name.startswith('~$')],   # skip Excel lock files
            key=lambda f: f.name
        )
        return [(i, f.name, str(f)) for i, f in enumerate(files, 1)]

    @staticmethod
    def select_input_file() -> str:
        Colors.step(1, "SELECT INPUT FILE")

        while True:
            folder = input(
                f"\n{Colors.CYAN}  Enter folder path containing your data file: {Colors.ENDC}"
            ).strip().strip('"\'')
            if os.path.isdir(folder):
                break
            Colors.error("Path not found — please try again.")

        files = FileManager.list_files(folder)
        if not files:
            Colors.error(f"No supported files (.xlsx/.xls/.csv) found in:\n  {folder}")
            sys.exit(1)

        print(f"\n{Colors.BOLD}  Available files:{Colors.ENDC}")
        for num, name, _ in files:
            print(f"    {Colors.GREEN}[{num}]{Colors.ENDC}  {name}")

        while True:
            try:
                sel = int(input(f"\n{Colors.CYAN}  Select file number: {Colors.ENDC}"))
                if 1 <= sel <= len(files):
                    Colors.success(f"Selected: {files[sel-1][1]}")
                    return files[sel-1][2]
                Colors.error(f"Enter a number between 1 and {len(files)}")
            except ValueError:
                Colors.error("Please enter a valid integer.")

    @staticmethod
    def select_output_folder() -> str:
        Colors.step(2, "SELECT OUTPUT FOLDER")

        while True:
            folder = input(
                f"\n{Colors.CYAN}  Enter output folder path "
                f"(created automatically if absent): {Colors.ENDC}"
            ).strip().strip('"\'')
            if folder:
                os.makedirs(folder, exist_ok=True)
                Colors.success(f"Output folder ready: {folder}")
                return folder
            Colors.error("Output path cannot be empty.")

    @staticmethod
    def load_file(file_path: str) -> pd.DataFrame:
        ext = Path(file_path).suffix.lower()
        try:
            if ext in ['.xlsx', '.xls']:
                xls = pd.ExcelFile(file_path)
                if len(xls.sheet_names) > 1:
                    print(f"\n{Colors.BOLD}  Available sheets:{Colors.ENDC}")
                    for i, s in enumerate(xls.sheet_names, 1):
                        print(f"    {Colors.GREEN}[{i}]{Colors.ENDC}  {s}")
                    while True:
                        try:
                            sel = int(input(f"\n{Colors.CYAN}  Select sheet: {Colors.ENDC}"))
                            if 1 <= sel <= len(xls.sheet_names):
                                sheet = xls.sheet_names[sel - 1]
                                df = pd.read_excel(file_path, sheet_name=sheet)
                                Colors.success(f"Loaded sheet: '{sheet}'")
                                break
                        except ValueError:
                            pass
                        Colors.error(f"Enter 1–{len(xls.sheet_names)}")
                else:
                    df = pd.read_excel(file_path)
                    Colors.success(f"Loaded: {Path(file_path).name}")

            elif ext == '.csv':
                df = pd.read_csv(file_path)
                Colors.success(f"Loaded: {Path(file_path).name}")
            else:
                Colors.error(f"Unsupported format: {ext}")
                sys.exit(1)

        except Exception as e:
            Colors.error(f"Could not read file: {e}")
            sys.exit(1)

        Colors.info(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
        return df


# ============================================================================
# CLASSIFIER
# ============================================================================
class MaterialClassifier:
    """Parses species/formula strings and assigns compound categories."""

    # Column names to try, in priority order (case-insensitive matching applied)
    _CANDIDATE_COLS = ['species', 'formula', 'FORMULA', 'composition']

    def __init__(self):
        self.db = ElementDatabase()
        self.parser = FormulaParser()

    # ------------------------------------------------------------------
    def _detect_column(self, df: pd.DataFrame) -> str:
        """Return the first matching candidate column, case-insensitively."""
        col_lower = {c.lower(): c for c in df.columns}
        for candidate in self._CANDIDATE_COLS:
            if candidate.lower() in col_lower:
                found = col_lower[candidate.lower()]
                Colors.info(f"Using column '{found}' for element extraction.")
                return found

        # Last resort: ask the user
        Colors.warning("Could not auto-detect a species/formula column.")
        Colors.info(f"Available columns: {', '.join(df.columns.tolist())}")
        while True:
            col = input(
                f"\n{Colors.CYAN}  Enter the column name to use: {Colors.ENDC}"
            ).strip()
            if col in df.columns:
                return col
            Colors.error(f"Column '{col}' not found. Please try again.")

    # ------------------------------------------------------------------
    def classify(self, elements: Set[str]) -> str:
        if not elements:
            return 'Unknown'

        metallic   = elements & self.db.METALS_AND_METALLOIDS

        # ── 1. Single element ──────────────────────────────────────────
        if len(elements) == 1:
            return 'Element'

        # ── 2. Carbide ─────────────────────────────────────────────────
        if 'C' in elements and metallic:
            return 'Carbide'

        # ── 3. Boride ──────────────────────────────────────────────────
        if 'B' in elements and metallic:
            return 'Boride'

        # ── 4. Nitride ─────────────────────────────────────────────────
        if 'N' in elements and metallic:
            return 'Nitride'

        # ── 5. Oxide ───────────────────────────────────────────────────
        if 'O' in elements:
            return 'Oxide'

        # ── 6. Halide ──────────────────────────────────────────────────
        if elements & self.db.HALOGENS:
            return 'Halide'

        # ── 7. Metal-Metalloid ─────────────────────────────────────────
        if elements.issubset(self.db.METALS_AND_METALLOIDS):
            return 'Metal-Metalloid'

        # ── 8. Other ───────────────────────────────────────────────────
        return 'Other'

    # ------------------------------------------------------------------
    def run(self, df: pd.DataFrame) -> pd.DataFrame:
        col = self._detect_column(df)
        df = df.copy()
        df['Elements'] = df[col].apply(self.parser.parse)
        df['Category'] = df['Elements'].apply(self.classify)
        return df


# ============================================================================
# EXPORTER
# ============================================================================
class Exporter:
    _CAT_COLOR: Dict[str, str] = {
        'Carbide':         Colors.RED,
        'Boride':          Colors.CYAN,
        'Nitride':         Colors.PINK,
        'Oxide':           Colors.YELLOW,
        'Halide':          Colors.LIME,
        'Metal-Metalloid': Colors.ORANGE,
        'Element':         Colors.WHITE,
        'Other':           Colors.DARK_BLUE,
        'Unknown':         Colors.GREY,
    }

    @staticmethod
    def save_categories(df: pd.DataFrame, output_dir: str) -> Dict[str, int]:
        summary: Dict[str, int] = {}
        export_df = df.drop(columns=['Elements', 'Category'])

        for cat in sorted(df['Category'].unique()):
            subset = export_df[df['Category'] == cat]
            out_path = os.path.join(output_dir, f"{cat}.xlsx")
            subset.to_excel(out_path, index=False)
            summary[cat] = len(subset)
            Colors.success(f"Saved  '{cat}.xlsx'  ({len(subset):,} rows)")

        return summary

    @staticmethod
    def print_summary(summary: Dict[str, int], total: int) -> None:
        Colors.header("CLASSIFICATION SUMMARY")

        col_w = max(len(k) for k in summary) + 2
        hdr = f"  {'Category':<{col_w}}  {'Count':>8}  {'Share':>7}   Distribution"
        print(f"{Colors.BOLD}{hdr}{Colors.ENDC}")
        print(f"  {'-'*col_w}  {'-'*8}  {'-'*7}   {'-'*42}")

        for cat, count in sorted(summary.items(), key=lambda x: -x[1]):
            pct     = count / total * 100
            bar_len = max(1, int(pct / 2.5))
            bar     = Colors.BAR_CHAR * bar_len
            col     = Exporter._CAT_COLOR.get(cat, Colors.CYAN)
            print(
                f"  {col}{cat:<{col_w}}{Colors.ENDC}"
                f"  {count:>8,}"
                f"  {pct:>6.1f}%"
                f"   {col}{bar}{Colors.ENDC}"
            )

        print(f"\n  {'─'*70}")
        print(f"  {Colors.BOLD}{'Total materials classified:':<{col_w + 2}}"
              f"  {total:>8,}{Colors.ENDC}")

    @staticmethod
    def print_file_list(output_dir: str, summary: Dict[str, int]) -> None:
        Colors.step("OUTPUT", "FILES CREATED")
        for cat in sorted(summary):
            path = os.path.join(output_dir, f"{cat}.xlsx")
            print(f"  {Colors.GREEN}→{Colors.ENDC}  {path}")


# ============================================================================
# MAIN
# ============================================================================
def main():
    Colors.header("MATERIAL COMPOUND CLASSIFIER  v1.0")

    input_path = FileManager.select_input_file()
    base_output_dir = FileManager.select_output_folder()
    output_dir = os.path.join(base_output_dir, "classified_data")
    os.makedirs(output_dir, exist_ok=True)

    Colors.info(f"All Excel files will be saved in: {output_dir}")

    Colors.step(3, "LOADING DATA")
    df = FileManager.load_file(input_path)

    Colors.step(4, "CLASSIFYING MATERIALS")
    classifier = MaterialClassifier()
    df_classified = classifier.run(df)

    cat_counts = df_classified['Category'].value_counts()
    Colors.info(
        f"Categories found ({len(cat_counts)}): "
        f"{', '.join(sorted(cat_counts.index.tolist()))}"
    )

    Colors.step(5, "EXPORTING CATEGORY FILES")
    summary = Exporter.save_categories(df_classified, output_dir)

    Exporter.print_summary(summary, total=len(df_classified))
    Exporter.print_file_list(output_dir, summary)

    Colors.header("CLASSIFICATION COMPLETE")


if __name__ == "__main__":
    main()


                       MATERIAL COMPOUND CLASSIFIER  v1.0                       


[STEP 1] SELECT INPUT FILE
--------------------------------------------------------------------------------



  Enter folder path containing your data file:  C:\



  Available files:
    [1]  AFLOW_cleaned_dataset.csv
    [2]  CRA_data.csv
    [3]  MP_cleaned_dataset.csv
    [4]  PCA_data_unclassified.csv
    [5]  SISSO_postprocessing_input_data.xlsx



  Select file number:  4


  ✓  Selected: PCA_data_unclassified.csv

[STEP 2] SELECT OUTPUT FOLDER
--------------------------------------------------------------------------------



  Enter output folder path (created automatically if absent):  C:\


  ✓  Output folder ready: C:\
  ℹ  All Excel files will be saved in: C:\classified_data

[STEP 3] LOADING DATA
--------------------------------------------------------------------------------
  ✓  Loaded: PCA_data_unclassified.csv
  ℹ  Rows: 11,509  |  Columns: 7

[STEP 4] CLASSIFYING MATERIALS
--------------------------------------------------------------------------------
  ℹ  Using column 'formula' for element extraction.
  ℹ  Categories found (8): Boride, Carbide, Element, Halide, Metal-Metalloid, Nitride, Other, Oxide

[STEP 5] EXPORTING CATEGORY FILES
--------------------------------------------------------------------------------
  ✓  Saved  'Boride.xlsx'  (412 rows)
  ✓  Saved  'Carbide.xlsx'  (462 rows)
  ✓  Saved  'Element.xlsx'  (294 rows)
  ✓  Saved  'Halide.xlsx'  (683 rows)
  ✓  Saved  'Metal-Metalloid.xlsx'  (6,582 rows)
  ✓  Saved  'Nitride.xlsx'  (535 rows)
  ✓  Saved  'Other.xlsx'  (1,413 rows)
  ✓  Saved  'Oxide.xlsx'  (1,128 rows)

                             CLASS